# Introdução à Orientação a Objetos em Java

Este notebook combina **teoria e prática** sobre classes, objetos, atributos, construtores, métodos, encapsulamento, referências e outros fundamentos de programação orientada a objetos.

## Objetivos

Ao final, você deverá ser capaz de:

- reconhecer abstração, encapsulamento, herança e polimorfismo;
- distinguir classes, objetos, tipos primitivos e tipos por referência;
- criar classes com atributos, construtores e métodos;
- aplicar modificadores de acesso e preservar invariantes;
- diferenciar membros de instância e membros estáticos;
- compreender escopo, `this`, `null` e passagem de argumentos por valor;
- criar representações textuais e objetos mutáveis ou imutáveis;
- separar regras de domínio da interação com o usuário;
- verificar o comportamento de uma classe com testes simples.

> Execute as células em ordem usando um kernel Java. As entradas de console são simuladas para que a execução completa não fique bloqueada esperando o teclado.


## 1. Tipos e modelos de domínio

Java possui **tipos primitivos**, como `int`, `double` e `boolean`, e **tipos por referência**, como classes, interfaces e arrays. Uma classe criada pelo programador define um novo tipo por referência e pode representar um conceito do domínio da aplicação.

| Categoria | Característica | Exemplos |
|---|---|---|
| Tipo primitivo | Armazena diretamente um valor básico | `int`, `double`, `boolean`, `char` |
| Tipo por referência | Armazena uma referência que permite alcançar um objeto | `String`, `Scanner`, `AccountNotebook` |

Modelar significa selecionar os dados e comportamentos relevantes para o problema. Uma conta bancária, por exemplo, pode ser representada por titular, saldo, depósito e saque, sem reproduzir todos os detalhes de um banco real.


In [1]:
int quantidade = 3;                  // tipo primitivo
String mensagem = "Olá, objetos";    // variável de tipo por referência

System.out.println("Quantidade: " + quantidade);
System.out.println(mensagem.toUpperCase());


Quantidade: 3
OL?, OBJETOS


## 2. Classes, objetos e os pilares da orientação a objetos

- **Classe:** define um tipo, descrevendo estado e comportamento.
- **Objeto:** é uma instância concreta de uma classe, com identidade e estado próprios.

Os quatro pilares fornecem um mapa conceitual:

| Pilar | Ideia central | Exemplo |
|---|---|---|
| Abstração | Representar apenas características relevantes | Uma conta oferece depósito e saque |
| Encapsulamento | Proteger o estado e controlar suas alterações | Saldo privado alterado por métodos |
| Herança | Criar um tipo mais específico a partir de outro | `EmailNotification extends Notification` |
| Polimorfismo | Tratar objetos diferentes por um tipo comum | Referência `Notification` apontando para `EmailNotification` |

Nesta aula, abstração e encapsulamento são aprofundados. O exemplo abaixo apenas antecipa herança e polimorfismo, que serão estudados posteriormente.


In [2]:
class Notification {
    public String send() {
        return "Notificação genérica";
    }
}

class EmailNotification extends Notification {
    @Override
    public String send() {
        return "E-mail enviado";
    }
}

Notification notification = new EmailNotification();
System.out.println(notification.send());


E-mail enviado


## 3. Anatomia de uma classe

Uma classe normalmente reúne:

- **atributos:** estado mantido pelo objeto;
- **construtores:** estabelecem o estado inicial;
- **métodos:** comportamentos e consultas oferecidos pelo objeto.

Em um projeto com arquivos `.java`, uma classe pública de nível superior deve estar em um arquivo com o mesmo nome:

```text
src/
├── Lamp.java      → public class Lamp
└── LampApp.java   → public class LampApp
```

```console
$ javac -d bin src/Lamp.java src/LampApp.java
$ java -cp bin LampApp
```

No notebook usamos classes não públicas, pois as células não correspondem a arquivos `.java` independentes.


In [3]:
class LampNotebook {
    private String color; // atributo
    private boolean on;

    public LampNotebook(String color) { // construtor
        this.color = color;
    }

    public void turnOn() { // método de instância
        on = true;
    }

    public boolean isOn() {
        return on;
    }

    public String getColor() {
        return color;
    }
}


In [4]:
LampNotebook yellowLamp = new LampNotebook("amarela");
LampNotebook blueLamp = new LampNotebook("azul");

yellowLamp.turnOn();

System.out.printf("Lâmpada %s: %s%n", yellowLamp.getColor(), yellowLamp.isOn() ? "acesa" : "apagada");
System.out.printf("Lâmpada %s: %s%n", blueLamp.getColor(), blueLamp.isOn() ? "acesa" : "apagada");


L?mpada amarela: acesa
L?mpada azul: apagada


## 4. Modificadores de acesso

Modificadores de acesso determinam de onde um tipo ou membro pode ser utilizado.

| Modificador | Acesso |
|---|---|
| `public` | Disponível para outras classes |
| `private` | Restrito à classe que declara o membro |
| sem modificador | Restrito às classes do mesmo pacote |
| `protected` | Disponível no pacote e, sob regras específicas, para subclasses |

Prefira atributos `private` e exponha como `public` somente a API necessária. O acesso direto abaixo não compila e permanece comentado de propósito:

```java
// yellowLamp.on = false; // erro: on has private access
```

`protected` será aprofundado junto com herança.


## 5. Atributos, valores padrão e escopo

Cada objeto possui seus próprios atributos de instância. Antes da execução do construtor, atributos recebem valores padrão: zero para números, `false` para `boolean` e `null` para referências.

O **escopo** indica onde um nome pode ser usado:

- atributo: corpo da classe e vida associada ao objeto;
- parâmetro: corpo do método ou construtor;
- variável local: bloco em que foi declarada.

Variáveis locais, ao contrário dos atributos, precisam ser inicializadas antes do uso. Um parâmetro pode ocultar um atributo de mesmo nome; `this` seleciona o atributo do objeto atual.


In [5]:
class ScopeNotebook {
    private int value; // começa com 0

    public ScopeNotebook(int value) {
        this.value = value; // atributo recebe o parâmetro
    }

    public int increment(int amount) { // parâmetro
        int previousValue = value;     // variável local
        value += amount;
        System.out.println("Valor anterior: " + previousValue);
        return value;
    }
}

ScopeNotebook scope = new ScopeNotebook(10);
System.out.println("Novo valor: " + scope.increment(5));


Valor anterior: 10
Novo valor: 15


## 6. Construtores, invariantes e políticas de validação

Construtores têm o mesmo nome da classe, não declaram retorno e executam durante `new`. Eles devem entregar um objeto em estado válido.

Uma **invariante** é uma condição que deve permanecer verdadeira para todo objeto válido. Para uma conta, podemos adotar `balance >= 0` e nome não vazio.

Há diferentes políticas para argumentos inválidos:

1. **normalizar:** substituir por um valor válido;
2. **ignorar:** manter o estado anterior;
3. **rejeitar:** lançar uma exceção.

A classe deve deixar sua política explícita e aplicá-la consistentemente.


In [6]:
class AccountNotebook {
    private String name;
    private double balance;
    private static int accountCount;

    public AccountNotebook(String name) {
        this(name, 0.0); // delegação para outro construtor
    }

    public AccountNotebook(String name, double balance) {
        this.name = normalizeName(name);
        this.balance = balance >= 0.0 ? balance : 0.0;
        accountCount++;
    }

    private static String normalizeName(String name) {
        return name != null && !name.isBlank() ? name : "Sem nome";
    }

    public String getName() {
        return name;
    }

    public void setName(String name) {
        if (name != null && !name.isBlank()) {
            this.name = name;
        }
    }

    public double getBalance() {
        return balance;
    }

    public void deposit(double amount) {
        if (amount > 0.0) {
            balance += amount;
        }
    }

    public boolean withdraw(double amount) {
        if (amount <= 0.0 || amount > balance) {
            return false;
        }
        balance -= amount;
        return true;
    }

    public static int getAccountCount() {
        return accountCount;
    }

    @Override
    public String toString() {
        return "AccountNotebook{name='%s', balance=%.2f}".formatted(name, balance);
    }
}


In [7]:
AccountNotebook anaAccount = new AccountNotebook("Ana", 100.0);
AccountNotebook unnamedAccount = new AccountNotebook("  ", -50.0);

System.out.println(anaAccount);
System.out.println(unnamedAccount);
System.out.println("Contas criadas: " + AccountNotebook.getAccountCount());


AccountNotebook{name='Ana', balance=100.00}
AccountNotebook{name='Sem nome', balance=0.00}
Contas criadas: 2


### Sobrecarga e delegação de construtores

`AccountNotebook(String)` e `AccountNotebook(String, double)` têm o mesmo nome e listas de parâmetros diferentes. Isso é **sobrecarga**. A chamada `this(name, 0.0)` delega a inicialização e deve ser a primeira instrução do construtor.

Métodos também podem ser sobrecarregados quando a quantidade, os tipos ou a ordem dos parâmetros diferem. Alterar somente o tipo de retorno não cria uma nova sobrecarga.


## 7. Rejeição de argumentos com IllegalArgumentException

Quando um argumento inválido representa erro de uso da classe, o construtor pode rejeitá-lo com `throw`. A execução é interrompida, a construção não é concluída e nenhuma referência válida é atribuída ao chamador.

`IllegalArgumentException` é uma exceção não verificada: o chamador pode capturá-la, mas não é obrigado a fazê-lo.


In [8]:
class TimeNotebook {
    private int hour;
    private int minute;
    private int second;

    public TimeNotebook(int hour, int minute, int second) {
        if (hour < 0 || hour >= 24
                || minute < 0 || minute >= 60
                || second < 0 || second >= 60) {
            throw new IllegalArgumentException("Hora, minuto ou segundo fora do intervalo");
        }
        this.hour = hour;
        this.minute = minute;
        this.second = second;
    }

    @Override
    public String toString() {
        return "%02d:%02d:%02d".formatted(hour, minute, second);
    }
}

System.out.println(new TimeNotebook(14, 30, 5));


14:30:05


In [9]:
try {
    TimeNotebook invalidTime = new TimeNotebook(25, 0, 0);
    System.out.println(invalidTime); // não é alcançado
} catch (IllegalArgumentException exception) {
    System.out.println("Construção rejeitada: " + exception.getMessage());
}


Constru??o rejeitada: Hora, minuto ou segundo fora do intervalo


## 8. Métodos de instância, métodos estáticos e sobrecarga

Um método de instância representa comportamento de um objeto, pode acessar seus atributos e possui `this`. Um método estático pertence à classe, é chamado pelo nome dela e não possui `this`.

```java
anaAccount.deposit(50.0);                 // instância
int total = AccountNotebook.getAccountCount(); // estático
```

Parâmetros generalizam o comportamento e `return` devolve um resultado ao chamador. Um método `void` executa uma ação sem produzir um valor de retorno.


In [10]:
class MethodNotebook {
    public static int square(int value) {
        return value * value;
    }

    public static double square(double value) {
        return value * value;
    }

    public static double maximum(double x, double y, double z) {
        return Math.max(x, Math.max(y, z));
    }
}

System.out.println("Quadrado inteiro: " + MethodNotebook.square(7));
System.out.println("Quadrado decimal: " + MethodNotebook.square(7.5));
System.out.println("Máximo: " + MethodNotebook.maximum(4.5, 9.0, 2.0));


Quadrado inteiro: 49
Quadrado decimal: 56.25
M?ximo: 9.0


## 9. Encapsulamento e métodos de acesso

Encapsular não significa apenas tornar atributos privados. Significa controlar o acesso, concentrar regras na classe responsável e preservar invariantes em todos os métodos públicos.

- *getter*: oferece uma consulta controlada;
- *setter*: permite alteração controlada;
- operação de domínio: expressa intenção, como `deposit` ou `withdraw`.

Nem todo atributo precisa de getter e setter. `setBalance`, por exemplo, permitiria ignorar as regras de depósito e saque.


In [11]:
System.out.println("Saldo inicial: " + anaAccount.getBalance());

anaAccount.deposit(50.0);
System.out.println("Após depósito: " + anaAccount.getBalance());

boolean withdrew = anaAccount.withdraw(40.0);
System.out.println("Saque realizado? " + withdrew);
System.out.println("Saldo final: " + anaAccount.getBalance());

anaAccount.deposit(-100.0); // ignorado: preserva a invariante
System.out.println("Após depósito inválido: " + anaAccount.getBalance());


Saldo inicial: 100.0
Ap?s dep?sito: 150.0
Saque realizado? true
Saldo final: 110.0
Ap?s dep?sito inv?lido: 110.0


## 10. Referências, identidade e null

Uma variável de tipo por referência permite alcançar um objeto. Atribuir uma referência a outra variável não copia o objeto: ambas podem apontar para a mesma instância.

`null` representa ausência de referência. Invocar um método por meio de `null` provoca `NullPointerException`.


In [12]:
AccountNotebook firstReference = new AccountNotebook("Bia", 200.0);
AccountNotebook secondReference = firstReference;

secondReference.deposit(25.0);
System.out.println(firstReference.getBalance()); // observa o mesmo objeto
System.out.println("Mesmo objeto? " + (firstReference == secondReference));

AccountNotebook absentAccount = null;
System.out.println("Referência ausente? " + (absentAccount == null));
// absentAccount.deposit(10.0); // provocaria NullPointerException


225.0
Mesmo objeto? true
Refer?ncia ausente? true


## 11. Passagem de argumentos por valor

Java sempre copia o valor fornecido como argumento:

- para um primitivo, copia o valor primitivo;
- para um objeto, copia o valor da referência.

Método e chamador podem alcançar o mesmo objeto, portanto uma alteração no objeto pode ser observada pelo chamador. Entretanto, atribuir outra referência ao parâmetro não altera a variável original.

> Java não passa objetos por referência; passa por valor uma referência ao objeto.


In [15]:
class ReferenceNotebook {
    public static void rename(AccountNotebook account) {
        account.setName("Nome alterado"); // altera o objeto compartilhado
    }

    public static void replace(AccountNotebook account) {
        account = new AccountNotebook("Outra conta", 999.0); // altera apenas o parâmetro
    }
}

AccountNotebook originalAccount = new AccountNotebook("Carlos", 80.0);
ReferenceNotebook.rename(originalAccount);
System.out.println(originalAccount.getName());

ReferenceNotebook.replace(originalAccount);
System.out.println(originalAccount); // continua sendo o objeto original


Nome alterado
AccountNotebook{name='Nome alterado', balance=80.00}


## 12. Objetos em uma aplicação console

Separe responsabilidades:

- a classe de domínio mantém estado e regras;
- a aplicação coleta entradas, cria objetos e apresenta resultados.

O exemplo simula o teclado com uma `String`. Para uma aplicação real, troque por `new Scanner(System.in)`. A simulação torna a célula reproduzível e evita bloqueio durante a execução automática.


In [16]:
import java.util.Locale;
import java.util.Scanner;

String simulatedInput = "Daniela\n100.00\n35.50\n";
Scanner input = new Scanner(simulatedInput).useLocale(Locale.US);

String accountName = input.nextLine();
double initialBalance = input.nextDouble();
double depositAmount = input.nextDouble();

AccountNotebook consoleAccount = new AccountNotebook(accountName, initialBalance);
consoleAccount.deposit(depositAmount);

System.out.printf(Locale.US, "Titular: %s; saldo: %.2f%n",
        consoleAccount.getName(), consoleAccount.getBalance());

input.close();


Titular: Daniela; saldo: 135.50


## 13. Representação textual com toString

Toda classe herda `toString()` de `Object`. Ao imprimir ou concatenar um objeto, Java solicita essa representação textual.

`@Override` informa ao compilador que o método redefine um método herdado. Uma representação útil auxilia diagnóstico e testes, mas não deve expor senhas ou dados sensíveis.


In [17]:
class StudentNotebook {
    private String name;
    private int age;
    private String course;

    public StudentNotebook(String name, int age, String course) {
        this.name = name != null && !name.isBlank() ? name : "Sem nome";
        this.age = age >= 0 ? age : 0;
        this.course = course != null && !course.isBlank() ? course : "Não informado";
    }

    public String getName() { return name; }
    public int getAge() { return age; }
    public String getCourse() { return course; }

    @Override
    public String toString() {
        return "StudentNotebook{name='%s', age=%d, course='%s'}"
                .formatted(name, age, course);
    }
}

StudentNotebook student = new StudentNotebook("Eva", 20, "Direito");
System.out.println(student);


StudentNotebook{name='Eva', age=20, course='Direito'}


## 14. Objetos mutáveis e imutáveis

- Um objeto **mutável** pode mudar após a construção. `AccountNotebook` muda com depósitos e saques.
- Um objeto **imutável** mantém o mesmo estado após a construção. Seus atributos podem ser `final`, e a classe não oferece setters.

Imutabilidade reduz os estados possíveis e facilita o raciocínio. Declarar a referência como `final` impede sua reatribuição, mas não torna automaticamente o objeto referenciado imutável.


In [18]:
final class ImmutableStudentNotebook {
    private final String name;
    private final int age;

    public ImmutableStudentNotebook(String name, int age) {
        if (name == null || name.isBlank() || age < 0) {
            throw new IllegalArgumentException("Nome ou idade inválidos");
        }
        this.name = name;
        this.age = age;
    }

    public String getName() { return name; }
    public int getAge() { return age; }

    @Override
    public String toString() {
        return "%s (%d anos)".formatted(name, age);
    }
}

ImmutableStudentNotebook immutableStudent = new ImmutableStudentNotebook("Fábio", 21);
System.out.println(immutableStudent);


F?bio (21 anos)


## 15. Valores monetários e BigDecimal

`double` é adequado para muitos cálculos científicos, mas não representa exatamente todos os valores decimais. Em aplicações financeiras reais, prefira `BigDecimal` construído a partir de texto.

```java
System.out.println(0.1 + 0.2); // 0.30000000000000004
```

`BigDecimal` é imutável: métodos como `add` retornam um novo valor.


In [19]:
import java.math.BigDecimal;

class MoneyAccountNotebook {
    private BigDecimal balance;

    public MoneyAccountNotebook(BigDecimal initialBalance) {
        if (initialBalance == null || initialBalance.signum() < 0) {
            throw new IllegalArgumentException("Saldo inicial inválido");
        }
        balance = initialBalance;
    }

    public void deposit(BigDecimal amount) {
        if (amount == null || amount.signum() <= 0) {
            throw new IllegalArgumentException("Depósito inválido");
        }
        balance = balance.add(amount);
    }

    public BigDecimal getBalance() {
        return balance;
    }
}

MoneyAccountNotebook moneyAccount = new MoneyAccountNotebook(new BigDecimal("100.00"));
moneyAccount.deposit(new BigDecimal("10.50"));
System.out.println("Saldo monetário: " + moneyAccount.getBalance());


Saldo monet?rio: 110.50


## 16. Verificação das regras da classe

Testes devem observar comportamentos públicos e confirmar invariantes em casos válidos, inválidos e de fronteira.

As instruções `assert` da linguagem ficam desabilitadas por padrão e exigem `java -ea` em uma aplicação tradicional. Para tornar a demonstração independente dessa opção, usamos um pequeno método `check` que lança `AssertionError` quando a condição é falsa.


In [20]:
class TestNotebook {
    public static void check(boolean condition, String message) {
        if (!condition) {
            throw new AssertionError(message);
        }
    }
}

AccountNotebook testedAccount = new AccountNotebook("Gabi", 100.0);
testedAccount.deposit(50.0);
TestNotebook.check(testedAccount.getBalance() == 150.0, "depósito positivo");

testedAccount.deposit(-20.0);
TestNotebook.check(testedAccount.getBalance() == 150.0, "depósito negativo deve ser ignorado");

TestNotebook.check(!testedAccount.withdraw(200.0), "saque superior ao saldo deve falhar");
TestNotebook.check(testedAccount.getBalance() == 150.0, "saque inválido não altera saldo");

System.out.println("Todos os testes passaram.");


Todos os testes passaram.


## 17. Prática guiada — retângulo

Implemente e experimente uma classe `RectangleNotebook` com as seguintes regras:

- largura e altura devem ser positivas;
- valores inválidos são normalizados para `1.0`;
- dimensões podem ser consultadas, mas não alteradas diretamente;
- métodos calculam área e perímetro.

Antes de executar a próxima célula, identifique onde aparecem abstração, encapsulamento, construtor, invariante, getters e métodos com retorno.


In [21]:
class RectangleNotebook {
    private double width;
    private double height;

    public RectangleNotebook(double width, double height) {
        this.width = width > 0.0 ? width : 1.0;
        this.height = height > 0.0 ? height : 1.0;
    }

    public double getWidth() { return width; }
    public double getHeight() { return height; }

    public double calculateArea() {
        return width * height;
    }

    public double calculatePerimeter() {
        return 2.0 * (width + height);
    }
}

RectangleNotebook rectangle = new RectangleNotebook(5.0, 3.0);
RectangleNotebook normalizedRectangle = new RectangleNotebook(-2.0, 4.0);

System.out.printf("Área: %.2f; perímetro: %.2f%n",
        rectangle.calculateArea(), rectangle.calculatePerimeter());
System.out.printf("Dimensões normalizadas: %.2f × %.2f%n",
        normalizedRectangle.getWidth(), normalizedRectangle.getHeight());


?rea: 15.00; per?metro: 16.00
Dimens?es normalizadas: 1.00 ? 4.00


## 18. Prática guiada — termômetro

O próximo exemplo exercita sobrecarga, delegação com `this(...)`, método de acesso, mutabilidade controlada e rejeição de valores abaixo do zero absoluto (`-273.15 °C`).

Experimente criar objetos pelos dois construtores e alterar a temperatura com valores válidos e inválidos.


In [22]:
class ThermometerNotebook {
    private double temperature;

    public ThermometerNotebook() {
        this(0.0);
    }

    public ThermometerNotebook(double temperature) {
        setTemperature(temperature);
    }

    public void setTemperature(double temperature) {
        if (temperature < -273.15) {
            throw new IllegalArgumentException("Temperatura abaixo do zero absoluto");
        }
        this.temperature = temperature;
    }

    public double getTemperature() {
        return temperature;
    }

    public double toFahrenheit() {
        return temperature * 9.0 / 5.0 + 32.0;
    }
}

ThermometerNotebook thermometer = new ThermometerNotebook(25.0);
System.out.printf("%.2f °C = %.2f °F%n",
        thermometer.getTemperature(), thermometer.toFahrenheit());


25.00 ?C = 77.00 ?F


## Referências oficiais

- Oracle. [Java Language Specification — Classes](https://docs.oracle.com/javase/specs/jls/se21/html/jls-8.html).
- Oracle. [Java Language Specification — Names and Scope](https://docs.oracle.com/javase/specs/jls/se21/html/jls-6.html).
- Oracle. [Java Language Specification — Method Invocation Expressions](https://docs.oracle.com/javase/specs/jls/se21/html/jls-15.html#jls-15.12).
- Oracle. [Java Language Specification — Class Instance Creation Expressions](https://docs.oracle.com/javase/specs/jls/se21/html/jls-15.html#jls-15.9).
- Oracle. [Object.toString — Java SE 21](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Object.html#toString()).
- Oracle. [IllegalArgumentException — Java SE 21](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/IllegalArgumentException.html).
- Oracle. [BigDecimal — Java SE 21](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/math/BigDecimal.html).


## Síntese

- Classes definem tipos; objetos são suas instâncias.
- Abstração seleciona características relevantes; encapsulamento protege estado e regras.
- Herança especializa tipos; polimorfismo permite tratá-los por uma abstração comum.
- Atributos, parâmetros e variáveis locais possuem escopos e tempos de vida diferentes.
- Construtores estabelecem o estado inicial e podem normalizar ou rejeitar argumentos.
- `this` referencia o objeto atual e `this(...)` delega entre construtores.
- Métodos de instância operam sobre objetos; métodos e atributos estáticos pertencem à classe.
- Sobrecarga usa o mesmo nome com listas de parâmetros diferentes.
- Getters, setters e operações de domínio devem preservar invariantes.
- Referências podem compartilhar um objeto; `null` representa ausência de referência.
- Java sempre passa argumentos por valor, inclusive valores de referência.
- `toString`, imutabilidade, tipos monetários adequados e testes melhoram a robustez da modelagem.
- Classes de domínio e aplicações de entrada e saída devem possuir responsabilidades distintas.
